In [13]:
import pandas as pd
import numpy as np

# 1. RUTES
ruta_base = '../data/processed/03_master_dataset_final_21_04_2026.csv'
ruta_crb = '../data/processed/02_dades_CRB_SeccioCensal.xlsx'
ruta_final = '../data/processed/04_Master_Dataset_Final_Integrat.xlsx'

print("📂 Carregant datasets...")
df_base = pd.read_csv(ruta_base, sep=';', encoding='utf-8')
df_crb = pd.read_excel(ruta_crb)

# 2. PETITA XARXA DE SEGURETAT PER L'EXCEL
def posar_zero_inicial(codi):
    codi_str = str(codi).strip().replace('.0', '')
    if len(codi_str) == 9:  # Si l'Excel s'ha menjat el 0 de '08019...'
        return '0' + codi_str
    return codi_str

df_base['CODI_UNIC'] = df_base['CODI_UNIC'].apply(posar_zero_inicial)
df_crb['CODI_UNIC'] = df_crb['CODI_UNIC'].apply(posar_zero_inicial)

# 3. EL MEGA-MERGE
print("🔗 Unint dades...")
master_df = pd.merge(df_base, df_crb, on='CODI_UNIC', how='left')

# 4. IMPUTACIÓ DE BITS (Tobler)
columnes_noves = df_crb.columns.drop('CODI_UNIC')
for col in columnes_noves:
    nuls = master_df[col].isna().sum()
    if nuls > 0:
        print(f"⚠️ Imputant {nuls} valors a: {col}")
        master_df['dist_temp'] = master_df['CODI_UNIC'].str[5:7] 
        master_df[col] = master_df[col].fillna(master_df.groupby('dist_temp')[col].transform('mean'))
        master_df.drop(columns=['dist_temp'], inplace=True)

# 5. EXPORTAR
print(f"💾 Guardant a: {ruta_final}")
master_df.to_excel(ruta_final, index=False)
print("✅ FET!")

📂 Carregant datasets...
🔗 Unint dades...
💾 Guardant a: ../data/processed/04_Master_Dataset_Final_Integrat.xlsx
✅ FET!
